# Assignment 07 - Bayesian Networks
**Author:** Wiktor Sosnowski, 348 561  
**Course:** Introduction to Artificial Intelligence (WSI)  
**Warsaw University of Technology**

## Imports

In [ ]:
# Author: Wiktor Sosnowski, 348 561

import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Output directories
RESULTS_DIR = os.path.join('..', '..', 'data', 'assignment_07', 'processed')
PLOTS_DIR   = os.path.join('..', '..', 'reports', 'assignment_07', 'plots')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,   exist_ok=True)

# Decimal separator used in saved CSV files
DECIMAL = ','

print('Directories ready.')

ModuleNotFoundError: No module named 'pgmpy'

## Bayesian Network - model

In [ ]:
def build_model(
    p_in_office: float = 0.40,
    p_light_on_given_in:  float = 0.50,
    p_light_on_given_out: float = 0.05,
    p_teams_on_given_in:  float = 0.80,
    p_teams_on_given_out: float = 0.05,
) -> VariableElimination:
    """
    Build and return a VariableElimination inference engine
    for the Bayesian network described in the task.

    Variables (state 0 = True/active, state 1 = False/inactive):
        Location : 0 = in office,  1 = remote
        Light    : 0 = on,         1 = off
        Teams    : 0 = logged in,  1 = logged out

    Parameters
    ----------
    p_in_office            : P(Location = in office)
    p_light_on_given_in    : P(Light = on  | Location = in office)
    p_light_on_given_out   : P(Light = on  | Location = remote)
    p_teams_on_given_in    : P(Teams = on  | Location = in office)
    p_teams_on_given_out   : P(Teams = on  | Location = remote)
    """
    model = DiscreteBayesianNetwork([
        ('Location', 'Light'),
        ('Location', 'Teams'),
    ])

    # P(Location)
    cpd_location = TabularCPD(
        variable='Location',
        variable_card=2,
        values=[[p_in_office], [1.0 - p_in_office]],
    )

    # P(Light | Location)
    cpd_light = TabularCPD(
        variable='Light',
        variable_card=2,
        values=[
            [p_light_on_given_in,         p_light_on_given_out],
            [1.0 - p_light_on_given_in,   1.0 - p_light_on_given_out],
        ],
        evidence=['Location'],
        evidence_card=[2],
    )

    # P(Teams | Location)
    cpd_teams = TabularCPD(
        variable='Teams',
        variable_card=2,
        values=[
            [p_teams_on_given_in,         p_teams_on_given_out],
            [1.0 - p_teams_on_given_in,   1.0 - p_teams_on_given_out],
        ],
        evidence=['Location'],
        evidence_card=[2],
    )

    model.add_cpds(cpd_location, cpd_light, cpd_teams)
    assert model.check_model(), 'Model is invalid.'

    return VariableElimination(model)


def query_light_on(
    infer: VariableElimination,
    evidence: dict | None = None,
) -> float:
    """
    Return P(Light = on) optionally conditioned on evidence.
    State 0 of 'Light' corresponds to light being on.
    """
    result = infer.query(['Light'], evidence=evidence, show_progress=False)
    return float(result.values[0])


print('Model functions defined.')

## Baseline - base model query

Parameters from the task:
- P(Location = in office) = 0.40
- P(Light = on | in office) = 0.50
- P(Light = on | remote)    = 0.05
- P(Teams = on | in office) = 0.80
- P(Teams = on | remote)    = 0.05

In [ ]:
BASE_PARAMS = dict(
    p_in_office           = 0.40,
    p_light_on_given_in   = 0.50,
    p_light_on_given_out  = 0.05,
    p_teams_on_given_in   = 0.80,
    p_teams_on_given_out  = 0.05,
)

base_infer = build_model(**BASE_PARAMS)

p_light_prior        = query_light_on(base_infer)
p_light_given_teams  = query_light_on(base_infer, evidence={'Teams': 0})

print(f'P(Light = on)              = {p_light_prior:.4f}')
print(f'P(Light = on | Teams = on) = {p_light_given_teams:.4f}')

## Experiment 1 - influence of P(Location = in office)

Fixed parameters:
- P(Light = on | in office) = 0.50
- P(Light = on | remote)    = 0.05
- P(Teams = on | in office) = 0.80
- P(Teams = on | remote)    = 0.05

Varied: P(Location = in office) in {0.10, 0.20, 0.40, 0.60, 0.80}

In [ ]:
exp1_values = [0.10, 0.20, 0.40, 0.60, 0.80]
exp1_rows = []

for val in exp1_values:
    params = {**BASE_PARAMS, 'p_in_office': val}
    infer  = build_model(**params)
    prior  = query_light_on(infer)
    post   = query_light_on(infer, evidence={'Teams': 0})
    exp1_rows.append({
        'P(in_office)':               val,
        'P(Light=on)':                round(prior, 4),
        'P(Light=on | Teams=on)':     round(post,  4),
        'Roznica':                    round(post - prior, 4),
    })

df_exp1 = pd.DataFrame(exp1_rows)
print(df_exp1.to_string(index=False))

# Save results
df_exp1.to_csv(
    os.path.join(RESULTS_DIR, 'exp1_p_in_office.csv'),
    index=False, decimal=DECIMAL, sep=';'
)

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_exp1['P(in_office)'], df_exp1['P(Light=on)'],
        marker='o', label='P(Light=on) - prior')
ax.plot(df_exp1['P(in_office)'], df_exp1['P(Light=on | Teams=on)'],
        marker='s', label='P(Light=on | Teams=on)')
ax.axvline(x=0.40, color='gray', linestyle='--', linewidth=0.8, label='wartość bazowa (0,40)')
ax.set_xlabel('P(Location = in office)')
ax.set_ylabel('P(Light = on)')
ax.set_title(
    'Eksperyment 1\n'
    'P(Light=on) vs P(in office)\n'
    'pozostałe parametry: bazowe'
)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'exp1_p_in_office.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved: {plot_path}')

## Experiment 2 - influence of P(Light = on | in office)

Fixed parameters:
- P(Location = in office)   = 0.40
- P(Light = on | remote)    = 0.05
- P(Teams = on | in office) = 0.80
- P(Teams = on | remote)    = 0.05

Varied: P(Light = on | in office) in {0.10, 0.25, 0.50, 0.75, 0.90}

In [ ]:
exp2_values = [0.10, 0.25, 0.50, 0.75, 0.90]
exp2_rows = []

for val in exp2_values:
    params = {**BASE_PARAMS, 'p_light_on_given_in': val}
    infer  = build_model(**params)
    prior  = query_light_on(infer)
    post   = query_light_on(infer, evidence={'Teams': 0})
    exp2_rows.append({
        'P(Light=on | in office)':    val,
        'P(Light=on)':                round(prior, 4),
        'P(Light=on | Teams=on)':     round(post,  4),
        'Roznica':                    round(post - prior, 4),
    })

df_exp2 = pd.DataFrame(exp2_rows)
print(df_exp2.to_string(index=False))

df_exp2.to_csv(
    os.path.join(RESULTS_DIR, 'exp2_p_light_given_in.csv'),
    index=False, decimal=DECIMAL, sep=';'
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_exp2['P(Light=on | in office)'], df_exp2['P(Light=on)'],
        marker='o', label='P(Light=on) - prior')
ax.plot(df_exp2['P(Light=on | in office)'], df_exp2['P(Light=on | Teams=on)'],
        marker='s', label='P(Light=on | Teams=on)')
ax.axvline(x=0.50, color='gray', linestyle='--', linewidth=0.8, label='wartość bazowa (0,50)')
ax.set_xlabel('P(Light = on | Location = in office)')
ax.set_ylabel('P(Light = on)')
ax.set_title(
    'Eksperyment 2\n'
    'P(Light=on) vs P(Light=on | in office)\n'
    'pozostałe parametry: bazowe'
)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'exp2_p_light_given_in.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved: {plot_path}')

## Experiment 3 - influence of P(Teams = on | in office)

Fixed parameters:
- P(Location = in office)   = 0.40
- P(Light = on | in office) = 0.50
- P(Light = on | remote)    = 0.05
- P(Teams = on | remote)    = 0.05

Varied: P(Teams = on | in office) in {0.20, 0.40, 0.60, 0.80, 0.95}

In [ ]:
exp3_values = [0.20, 0.40, 0.60, 0.80, 0.95]
exp3_rows = []

for val in exp3_values:
    params = {**BASE_PARAMS, 'p_teams_on_given_in': val}
    infer  = build_model(**params)
    prior  = query_light_on(infer)
    post   = query_light_on(infer, evidence={'Teams': 0})
    exp3_rows.append({
        'P(Teams=on | in office)':    val,
        'P(Light=on)':                round(prior, 4),
        'P(Light=on | Teams=on)':     round(post,  4),
        'Roznica':                    round(post - prior, 4),
    })

df_exp3 = pd.DataFrame(exp3_rows)
print(df_exp3.to_string(index=False))

df_exp3.to_csv(
    os.path.join(RESULTS_DIR, 'exp3_p_teams_given_in.csv'),
    index=False, decimal=DECIMAL, sep=';'
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_exp3['P(Teams=on | in office)'], df_exp3['P(Light=on)'],
        marker='o', label='P(Light=on) - prior')
ax.plot(df_exp3['P(Teams=on | in office)'], df_exp3['P(Light=on | Teams=on)'],
        marker='s', label='P(Light=on | Teams=on)')
ax.axvline(x=0.80, color='gray', linestyle='--', linewidth=0.8, label='wartość bazowa (0,80)')
ax.set_xlabel('P(Teams = on | Location = in office)')
ax.set_ylabel('P(Light = on)')
ax.set_title(
    'Eksperyment 3\n'
    'P(Light=on) vs P(Teams=on | in office)\n'
    'pozostałe parametry: bazowe'
)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'exp3_p_teams_given_in.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved: {plot_path}')

## Experiment 4 - influence of P(Teams = on | remote)

Fixed parameters:
- P(Location = in office)   = 0.40
- P(Light = on | in office) = 0.50
- P(Light = on | remote)    = 0.05
- P(Teams = on | in office) = 0.80

Varied: P(Teams = on | remote) in {0.01, 0.05, 0.15, 0.30, 0.50}

In [ ]:
exp4_values = [0.01, 0.05, 0.15, 0.30, 0.50]
exp4_rows = []

for val in exp4_values:
    params = {**BASE_PARAMS, 'p_teams_on_given_out': val}
    infer  = build_model(**params)
    prior  = query_light_on(infer)
    post   = query_light_on(infer, evidence={'Teams': 0})
    exp4_rows.append({
        'P(Teams=on | remote)':       val,
        'P(Light=on)':                round(prior, 4),
        'P(Light=on | Teams=on)':     round(post,  4),
        'Roznica':                    round(post - prior, 4),
    })

df_exp4 = pd.DataFrame(exp4_rows)
print(df_exp4.to_string(index=False))

df_exp4.to_csv(
    os.path.join(RESULTS_DIR, 'exp4_p_teams_given_out.csv'),
    index=False, decimal=DECIMAL, sep=';'
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_exp4['P(Teams=on | remote)'], df_exp4['P(Light=on)'],
        marker='o', label='P(Light=on) - prior')
ax.plot(df_exp4['P(Teams=on | remote)'], df_exp4['P(Light=on | Teams=on)'],
        marker='s', label='P(Light=on | Teams=on)')
ax.axvline(x=0.05, color='gray', linestyle='--', linewidth=0.8, label='wartość bazowa (0,05)')
ax.set_xlabel('P(Teams = on | Location = remote)')
ax.set_ylabel('P(Light = on)')
ax.set_title(
    'Eksperyment 4\n'
    'P(Light=on) vs P(Teams=on | remote)\n'
    'pozostałe parametry: bazowe'
)
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'exp4_p_teams_given_out.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Saved: {plot_path}')

## Save combined results to JSON

In [ ]:
all_results = {
    'baseline': {
        'params': BASE_PARAMS,
        'p_light_on_prior':       round(p_light_prior,       4),
        'p_light_on_given_teams': round(p_light_given_teams, 4),
    },
    'exp1_p_in_office':         exp1_rows,
    'exp2_p_light_given_in':    exp2_rows,
    'exp3_p_teams_given_in':    exp3_rows,
    'exp4_p_teams_given_out':   exp4_rows,
}

json_path = os.path.join(RESULTS_DIR, 'results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f'All results saved to {json_path}')